# 🩹 Maya Fills the Gaps: Day 1 Practical
### Week 3 · Handling Missing Data and Preparing Features

Nearly a third of the clinic's follow-up patients never come back. Dr. Kofi wants to predict who, so that Maya can phone them first.
Today you'll **find the gaps, fill them, turn words into numbers, scale the columns and train your first model**.

**How to use this notebook**
- Keep this notebook and **followup_visits.csv** in the same folder.
- Each step has a 📖 story, a 🎯 task and a ✏️ cell for your code.
- Press **Shift + Enter** to run a cell, and run the steps **in order**.
- Compare your result with the ✅ *expected output*. Your tables may look nicer than the plain text shown here.
- Muddled? Use **Kernel → Restart & Run All**.

> 🔒 All data in this notebook is made up. Never practise on real patient records.

---
## Step 1 · Meet the follow-up list  *(~4 min)*
**Concept:** features and target, .mean() of a 0/1 column

📖 Maya's charts worked: the district funded a follow-up programme. But many patients don't come back. Kofi asks: can we tell in advance who will miss their appointment, so Maya can phone them first?

🎯 **Your task**
1. Make sure **followup_visits.csv** is in the same folder as this notebook, then load it.
2. Count how many appointments were missed. The `missed` column holds 1 for missed and 0 for attended.
3. Look at the first 3 rows. Which column are we trying to predict? Which columns could help?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
import pandas as pd

df = pd.read_csv("followup_visits.csv")
print(df.shape)
print("Missed:", df["missed"].sum(), "of", len(df))
print("Missed rate:", round(df["missed"].mean() * 100, 1), "%")
df.head(3)
```
</details>

In [ ]:
# Step 1: load followup_visits.csv and meet the target
import pandas as pd


✅ **Expected output**
```
(720, 12)
Missed: 223 of 720
Missed rate: 31.0 %
  appt_id  age sex         ward  transport  distance_km  monthly_income  \
0   F0001   19   F  Outpatients       Walk          3.9           125.0   
1   F0002   25   F    Maternity  Motorbike          7.7           115.0   
2   F0003   34   F  Outpatients        Bus         26.5           345.0   

   days_waited  prev_missed has_phone sms_reminder  missed  
0           39            0       Yes          Yes       0  
1           39            0       Yes          Yes       0  
2           40            0       Yes          Yes       0
```
💡 The column we want to predict is the target (missed). The columns we predict from are the features. The mean of a 0/1 column is the share of 1s, so it gives the missed rate directly.

🆘 *If you see* `FileNotFoundError: followup_visits.csv` → The CSV isn't next to your notebook. Move it into the same folder and run the cell again.

---
## Step 2 · Find the gaps  *(~4 min)*
**Concept:** .isna(), .sum(), .mean(), .any(axis=1)

📖 In Week 2 we left the missing temperatures empty. A model can't do that: one empty cell and it refuses to work. First, Maya needs to know where the gaps are.

🎯 **Your task**
1. Count the missing values in each column.
2. Turn the counts into percentages.
3. Count the rows that have at least one gap anywhere.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
missing = df.isna().sum()
print(missing[missing > 0])

percent = (df.isna().mean() * 100).round(1)
print(percent[percent > 0])

print("Rows with at least one gap:", df.isna().any(axis=1).sum())
```
</details>

In [ ]:
# Step 2: where are the gaps?


✅ **Expected output**
```
transport          43
distance_km        65
monthly_income    101
dtype: int64
transport          6.0
distance_km        9.0
monthly_income    14.0
dtype: float64
Rows with at least one gap: 173
```
💡 isna() gives True wherever a value is missing. True counts as 1, so .sum() counts the gaps and .mean() gives the share. any(axis=1) asks the question across each row instead of down each column.

🆘 *If you see* `AttributeError: 'DataFrame' object has no attribute 'isnan'` → The pandas method is isna(), with no second n.

---
## Step 3 · Option 1: drop the rows (and count the cost)  *(~4 min)*
**Concept:** .dropna()

📖 The quick fix is to throw away every row that has a gap. Before Maya does that, she checks what it would cost.

🎯 **Your task**
1. Make a copy of the table with `dropna()` and compare the row counts.
2. Work out how many appointments were lost.
3. Compare the missed rate before and after. Did dropping rows change the picture?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
dropped = df.dropna()
print("Before:", len(df), " After dropna:", len(dropped))
print("Lost:", len(df) - len(dropped), "appointments")

print("Missed rate, all rows: ", round(df["missed"].mean() * 100, 1), "%")
print("Missed rate, kept rows:", round(dropped["missed"].mean() * 100, 1), "%")
```
</details>

In [ ]:
# Step 3: what would dropping cost?


✅ **Expected output**
```
Before: 720  After dropna: 547
Lost: 173 appointments
Missed rate, all rows:  31.0 %
Missed rate, kept rows: 32.0 %
```
💡 dropna() removes any row with even one gap, so three columns with a few gaps each add up to nearly a quarter of the table. Dropping is fine when only a handful of rows are affected. Here it wastes too much.

🆘 *If you see* `The row count didn't change` → dropna() gives back a new table. You need to store it: dropped = df.dropna().

---
## Step 4 · Option 2: fill number gaps with the median  *(~6 min)*
**Concept:** .fillna(), mean vs median

📖 Instead of deleting patients, Maya fills each gap with a typical value. But which is more typical: the mean or the median? A few high earners make a big difference.

🎯 **Your task**
1. Compare the mean and the median of `monthly_income`, and look at the highest value.
2. Make a working copy called `clean`, so that `df` stays untouched.
3. Fill the gaps in `monthly_income` and `distance_km` with each column's median.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
print("Income mean:  ", round(df["monthly_income"].mean(), 1))
print("Income median:", df["monthly_income"].median())
print("Highest income:", df["monthly_income"].max())

clean = df.copy()
clean["monthly_income"] = clean["monthly_income"].fillna(clean["monthly_income"].median())
clean["distance_km"] = clean["distance_km"].fillna(clean["distance_km"].median())
print(clean[["monthly_income", "distance_km"]].isna().sum())
```
</details>

In [ ]:
# Step 4: fill the number gaps


✅ **Expected output**
```
Income mean:   202.2
Income median: 170.0
Highest income: 1320.0
monthly_income    0
distance_km       0
dtype: int64
```
💡 A few very high incomes pull the mean upwards, but the median (the middle patient) hardly moves. For lopsided columns like income and distance, the median is the safer filler. Filling in a value like this is called imputation.

🆘 *If you see* `The gaps are still there` → fillna() gives back a new column. You need to assign it: clean["col"] = clean["col"].fillna(...).

---
## Step 5 · Fill category gaps with the most common value  *(~4 min)*
**Concept:** .value_counts(dropna=False), .mode()

📖 Transport is a word, not a number, so there's no median. For categories, Maya uses the most common answer.

🎯 **Your task**
1. Count the transport values, including the gaps, with `dropna=False`.
2. Find the most common value with `.mode()[0]`.
3. Fill the gaps, then confirm that no gaps are left anywhere in `clean`.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
print(clean["transport"].value_counts(dropna=False))

most_common = clean["transport"].mode()[0]
print("Most common:", most_common)

clean["transport"] = clean["transport"].fillna(most_common)
print("Gaps left in the whole table:", clean.isna().sum().sum())
```
</details>

In [ ]:
# Step 5: fill the category gaps


✅ **Expected output**
```
transport
Walk         259
Bus          203
Motorbike    146
Car           69
NaN           43
Name: count, dtype: int64
Most common: Walk
Gaps left in the whole table: 0
```
💡 mode() gives the most frequent value. It returns a list-like result, because there can be a tie, so [0] takes the first. The double .sum().sum() adds up the gaps per column, then adds those totals together.

🆘 *If you see* `KeyError: 0` → You wrote .mode[0]. mode is an action, so it needs brackets: .mode()[0].

---
## Step 6 · Create features that carry a clue  *(~5 min)*
**Concept:** boolean → 0/1, .astype(int), checking a feature against the target

📖 Maya knows her patients. Someone who has missed before often misses again, and long waits make people forget. She turns that knowledge into new columns. This is called feature engineering.

🎯 **Your task**
1. Create `missed_before`: 1 if the patient has missed at least one earlier appointment.
2. Create `long_wait`: 1 if the appointment was booked more than 30 days ahead.
3. Check each new feature against the target with `groupby`. Does it separate the two groups?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
clean["missed_before"] = (clean["prev_missed"] > 0).astype(int)
clean["long_wait"] = (clean["days_waited"] > 30).astype(int)

print(clean.groupby("missed_before")["missed"].mean().round(2))
print(clean.groupby("long_wait")["missed"].mean().round(2))
```
</details>

In [ ]:
# Step 6: two new features


✅ **Expected output**
```
missed_before
0    0.18
1    0.42
Name: missed, dtype: float64
long_wait
0    0.20
1    0.43
Name: missed, dtype: float64
```
💡 A comparison gives True/False, and astype(int) turns that into 1/0. The groupby is the test of a good feature: if the missed rate is very different between the groups, the feature carries a clue the model can use.

🆘 *If you see* `KeyError: 'prev_miss'` → Check the column name: it's prev_missed.

---
## Step 7 · Turn words into numbers  *(~6 min)*
**Concept:** .map(), pd.get_dummies

📖 A model only does maths, and you can't multiply "Bus" by anything. Yes/No columns become 1/0. Columns with several options get one new column per option.

🎯 **Your task**
1. Use `.map` to turn `has_phone` and `sms_reminder` into 1 and 0.
2. Use `pd.get_dummies` on `sex`, `ward` and `transport`. Store the result in `ready`.
3. Print the new column names. How many columns did `transport` turn into?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
clean["has_phone"] = clean["has_phone"].map({"Yes": 1, "No": 0})
clean["sms_reminder"] = clean["sms_reminder"].map({"Yes": 1, "No": 0})

ready = pd.get_dummies(clean, columns=["sex", "ward", "transport"], dtype=int)
print(ready.shape)
print(list(ready.columns[8:]))
ready[["appt_id", "transport_Bus", "transport_Car", "transport_Motorbike", "transport_Walk"]].head(3)
```
</details>

In [ ]:
# Step 7: encode the text columns (run this cell only once)


✅ **Expected output**
```
(720, 20)
['missed', 'missed_before', 'long_wait', 'sex_F', 'sex_M', 'ward_Maternity', 'ward_Outpatients', 'ward_Paediatrics', 'transport_Bus', 'transport_Car', 'transport_Motorbike', 'transport_Walk']
  appt_id  transport_Bus  transport_Car  transport_Motorbike  transport_Walk
0   F0001              0              0                    0               1
1   F0002              0              0                    1               0
2   F0003              1              0                    0               0
```
💡 Run this cell only once! This is one-hot encoding: each option gets its own column, with a 1 in the column that applies. We don't number the options 1, 2, 3, 4, because that would tell the model that Walk is 'four times' Bus.

🆘 *If you see* `has_phone is now all NaN` → You ran the cell twice, so map found no Yes/No left to translate. Use Kernel → Restart & Run All.

---
## Step 8 · Put the numbers on the same scale  *(~6 min)*
**Concept:** standardising by hand, StandardScaler, fit_transform

📖 Income runs into the hundreds. Previous misses run from 0 to 5. A model that measures distance between patients will only 'see' income, unless every column is put on the same scale.

🎯 **Your task**
1. Look at the mean, spread, min and max of the number columns.
2. Standardise `age` by hand: subtract the mean, divide by the standard deviation.
3. Let `StandardScaler` do all five columns, and check the new means and spreads.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.preprocessing import StandardScaler

num_cols = ["age", "distance_km", "monthly_income", "days_waited", "prev_missed"]
print(ready[num_cols].describe().loc[["mean", "std", "min", "max"]].round(1))

by_hand = (ready["age"] - ready["age"].mean()) / ready["age"].std()
print(by_hand.head(3).round(2).tolist())

scaler = StandardScaler()
scaled = ready.copy()
scaled[num_cols] = scaler.fit_transform(ready[num_cols])
print(scaled[num_cols].describe().loc[["mean", "std", "min", "max"]].round(1))
```
</details>

In [ ]:
# Step 8: scale the number columns


✅ **Expected output**
```
       age  distance_km  monthly_income  days_waited  prev_missed
mean  32.4          7.0           197.7         30.6          0.7
std   20.4          5.7           132.2         17.6          0.8
min    0.0          0.3            20.0          1.0          0.0
max   79.0         41.3          1320.0         60.0          5.0
[-0.65, -0.36, 0.08]
      age  distance_km  monthly_income  days_waited  prev_missed
mean -0.0          0.0            -0.0          0.0         -0.0
std   1.0          1.0             1.0          1.0          1.0
min  -1.6         -1.2            -1.3         -1.7         -0.9
max   2.3          6.0             8.5          1.7          5.1
```
💡 After scaling, every column has a mean of 0 and a spread of 1. A value of 2 now means 'well above average' in any column. fit works out each column's mean and spread, and transform applies them.

🆘 *If you see* `ModuleNotFoundError: No module named 'sklearn'` → scikit-learn isn't in this Python. Open Jupyter from Anaconda, which includes it.

---
## Step 9 · An imputer that remembers  *(~5 min)*
**Concept:** SimpleImputer, .fit_transform, .statistics_, .transform

📖 Next month, new patients will arrive with new gaps. scikit-learn's imputer learns the medians once, remembers them, and can fill any future table the same way.

🎯 **Your task**
1. Create a `SimpleImputer` with the median strategy.
2. Fit it on the two original columns from `df` and look at what it learned.
3. Give it a brand-new patient with both values missing. What does it fill in?

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
filled = imputer.fit_transform(df[["distance_km", "monthly_income"]])
print("Learned medians:", imputer.statistics_)
print("Gaps left:", pd.isna(filled).sum())

new_patient = pd.DataFrame({"distance_km": [None], "monthly_income": [None]}, dtype=float)
print("New patient filled with:", imputer.transform(new_patient))
```
</details>

In [ ]:
# Step 9: scikit-learn's imputer


✅ **Expected output**
```
Learned medians: [  5.2 170. ]
Gaps left: 0
New patient filled with: [[  5.2 170. ]]
```
💡 Every scikit-learn tool works this way. fit learns from the data, transform applies what was learned, and fit_transform does both. The medians are the same ones you found in Step 4. On Saturday we'll chain these tools into one pipeline.

🆘 *If you see* `ValueError: Expected 2D array, got 1D array` → Use double brackets, df[["distance_km", "monthly_income"]]. scikit-learn wants a table, even for one column.

---
## Step 10 · Mini-project: Maya's first model  *(~8 min)*
**Concept:** train_test_split, KNeighborsClassifier, .fit, .score

📖 Time to see if the preparation paid off. Maya trains a 'similar patients' model twice: once on the unscaled table and once on the scaled one. She hides a quarter of the patients to test it on.

🎯 **Your task**
1. List the feature columns: everything except `appt_id` and `missed`.
2. Split, train and score using the **unscaled** table `ready`.
3. Do the same with `scaled`. What difference did scaling make?
4. Save your prepared table.

<details><summary>👀 Need the code? Click to reveal (try on your own first!)</summary>

```python
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

features = [c for c in ready.columns if c not in ["appt_id", "missed"]]
y = ready["missed"]

X_train, X_test, y_train, y_test = train_test_split(ready[features], y, test_size=0.25, random_state=5)
model = KNeighborsClassifier()
model.fit(X_train, y_train)
print("Not scaled:", round(model.score(X_test, y_test) * 100, 1), "% correct")

X_train, X_test, y_train, y_test = train_test_split(scaled[features], y, test_size=0.25, random_state=5)
model = KNeighborsClassifier()
model.fit(X_train, y_train)
print("Scaled:    ", round(model.score(X_test, y_test) * 100, 1), "% correct")

scaled.to_csv("my_followup_prepared.csv", index=False)
print("Saved", scaled.shape[0], "rows and", scaled.shape[1], "columns")
```
</details>

In [ ]:
# Step 10: Maya's first model


✅ **Expected output**
```
Not scaled: 69.4 % correct
Scaled:     75.0 % correct
Saved 720 rows and 20 columns
```
💡 The model looks at the 5 most similar past patients and goes with the majority. fit is the studying and score is the exam, on patients the model has never seen. The only change between the two runs is the scaling.

🆘 *If you see* `ValueError: could not convert string to float` → A text column is still in your features. Check that Step 7 ran, and that appt_id is left out.

---
# 🌟 Finished early? Challenges
Optional extras for fast finishers.

### Challenge 1 · Flag the gaps before you fill them
In `df`, create a column `income_missing` that is 1 where `monthly_income` is missing and 0 otherwise. Then compare the missed rate of the two groups. Would this flag be a useful feature?

In [ ]:
# Challenge 1


### Challenge 2 · A different scaler (MinMaxScaler)
Use `MinMaxScaler` from `sklearn.preprocessing` on the five number columns of `ready`. Print the min and max of each column afterwards.

In [ ]:
# Challenge 2


### Challenge 3 · scikit-learn's one-hot encoder
Use `OneHotEncoder(sparse_output=False)` on the `transport` column of `clean`. Print the names of the columns it creates and the first 3 rows.

In [ ]:
# Challenge 3


---
## 🎉 Well done!
You prepared data for a model and trained your first one. **Save this notebook** (Ctrl + S), then commit it and push it to GitHub.

Next: **Week3_Exercises_Feature_Prep.ipynb**, then Saturday's class on cross-validation and pipelines.